<a href="https://colab.research.google.com/github/MalavMDesai/GenAIAssignment/blob/main/Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install langchain-community
!pip install pymupdf
!pip install langchain-google-genai
!pip install chromadb
!pip install langchain-openai

In [2]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_4963/2747336630.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_key')

# policy_file_url = "https://github.com/MalavMDesai/GenAIAssignment/raw/refs/heads/main/Assignment5_Policy.pdf"
policy_file_url="https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf"

loader = PyMuPDFLoader(policy_file_url)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

In [4]:
print(len(chunks))
print(chunks[0])

84
page_content='OPTIMA PLUS - Prospectus 
HDFC ERGO General Insurance Limited 
 
HDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146. CIN: U66030MH2007PLC177117. Registered & 
Corporate Office: 6th Floor, Leela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059. UIN: 
Optima Plus - HDHHLIP21336V022021  
 
1 
Optima Plus - Prospectus 
Eligibility 
▪ 
This policy covers persons in the age group 91 days to 65 years.  
▪ 
The maximum entry age is restricted upto 65 years. 
▪ 
Child between 91 days to 5 years can be insured only when either parent is getting 
insured under this policy.  
▪ 
The policy offers coverage on individual sum insured basis.  
▪ 
This policy can be issued to an individual and/or family. 
▪' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-05-06T10:28:39+05:30', 'source': 'https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf', 'fil

In [5]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", task_type="retrieval_document")
vector_store = Chroma.from_documents(chunks, embedding)
retriever = vector_store.as_retriever()

llm = ChatGoogleGenerativeAI(model="models/gemini-2.0-flash-exp", temperature=0.0, override_model_name=True)

system_prompt = (
    "You work as a expert Insurance Claims agent in HDFC ERGO General Insurance Company Limited \n"
    "Answer the questions using only provided policy context. \n"
    "If you do not know the answer or if it is not is the provided document than exactly say:\n"
    "'Unable to find the information in policy document, connect to customer care'"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', "{input}")
])

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query1 = "What is the waiting period for pre-existing diseases?"
res = rag_chain.invoke(query1)

print(f"User query: {query1}")
print(f"ans: \n {res}")


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 37.252730137s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDim